# Evaluación Parcial 1 - Machine Learning
**Asignatura:** MLY0100 Machine Learning  
**Integrantes:** Christian Sandoval, Nicolás Vega  
**Fecha:** 25 de septiembre de 2026

# Fase 1 - Comprensión del Negocio
El dataset contiene publicaciones de propiedades en venta en Argentina. El objetivo es analizar sus características y preparar los datos antes del modelado.

**Pregunta:** ¿Qué características, como ubicación, superficie y tipo de propiedad, se relacionan con las diferencias de precio?

**Supuestos:** los nulos no representan cero, los valores extremos se revisan antes de modificarlos y las conclusiones se limitan al dataset.

- **Target de regresión:** `price`, porque es numérico continuo.
- **Target de clasificación:** `property_type`, porque contiene clases como Departamento, Casa y PH.

En esta entrega no se entrenan modelos.


# Fase 2 - Comprensión de los Datos
## Carga e inspección
Se importan las librerías y se carga el CSV incluido en el ZIP. Luego se revisan dimensiones, tipos y estadísticas generales.


In [ ]:
%matplotlib inline
import os, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler

nombre_zip = 'Evaluación Parcial 1 - Dataset.zip'
ruta_zip = nombre_zip if os.path.exists(nombre_zip) else '../' + nombre_zip
if not os.path.exists(ruta_zip):
    from google.colab import files
    files.upload()
    ruta_zip = nombre_zip

with zipfile.ZipFile(ruta_zip) as z:
    nombre_csv = [n for n in z.namelist()
                  if n.endswith('DS1-18-Datos-Properati.csv')
                  and not n.startswith('__MACOSX')][0]
    with z.open(nombre_csv) as f:
        df = pd.read_csv(f)

print('Dimensiones:', df.shape)
print(df.dtypes)
df.info()
display(df.head())
display(df.describe(include='all').T)


El dataset tiene **146.660 filas y 19 columnas**. Las fechas están como texto y existen nulos en `lat`, `lon`, `bathrooms`, `surface_total` y `surface_covered`.

## Estadísticos descriptivos
Se calculan media, mediana, moda, desviación estándar, varianza e IQR para las variables numéricas más relevantes.


In [ ]:
variables = ['rooms','bedrooms','bathrooms','surface_total','surface_covered','price']
estadisticos = pd.DataFrame({
    'media': df[variables].mean(),
    'mediana': df[variables].median(),
    'moda': df[variables].mode().iloc[0],
    'std': df[variables].std(),
    'varianza': df[variables].var(),
    'Q1': df[variables].quantile(.25),
    'Q3': df[variables].quantile(.75)
})
estadisticos['IQR'] = estadisticos['Q3'] - estadisticos['Q1']
estadisticos.round(2)


En `price`, `surface_total` y `surface_covered` la media y mediana son bastante diferentes, señal de asimetría y posibles valores extremos.

## Distribuciones
Se usan histograma, boxplots, barras, scatter y heatmap. El percentil 99 solo se usa aquí para que los gráficos sean legibles.


In [ ]:
p99_price = df['price'].quantile(.99)
p99_superficie = df['surface_total'].quantile(.99)
precios = df.loc[df['price'] <= p99_price, 'price'].dropna()

plt.figure(figsize=(8,5))
plt.hist(precios, bins=40, edgecolor='black')
plt.axvline(precios.mean(), linestyle='--', label='Media')
plt.axvline(precios.median(), linestyle='-', label='Mediana')
plt.title('Distribución del precio'); plt.xlabel('Precio (USD)'); plt.ylabel('Cantidad'); plt.legend(); plt.show()

fig, ax = plt.subplots(1,2,figsize=(12,4))
ax[0].boxplot(precios, vert=False); ax[0].set_title('Boxplot de price'); ax[0].set_xlabel('Precio (USD)'); ax[0].set_ylabel('Distribución')
ax[1].boxplot(df.loc[df['surface_total']<=p99_superficie,'surface_total'].dropna(),vert=False)
ax[1].set_title('Boxplot de surface_total'); ax[1].set_xlabel('Superficie total (m²)'); ax[1].set_ylabel('Distribución')
plt.tight_layout(); plt.show()

tipos = df['property_type'].value_counts().head(10)
plt.figure(figsize=(9,5)); plt.barh(tipos.index, tipos.values)
plt.title('10 tipos de propiedad con más publicaciones'); plt.xlabel('Cantidad'); plt.ylabel('Tipo de propiedad')
plt.gca().invert_yaxis(); plt.show()

muestra = df[(df['surface_total']<=p99_superficie)&(df['price']<=p99_price)][['surface_total','price']].dropna().sample(5000,random_state=42)
plt.figure(figsize=(8,5)); plt.scatter(muestra['surface_total'],muestra['price'],alpha=.35)
plt.title('Superficie total y precio'); plt.xlabel('Superficie total (m²)'); plt.ylabel('Precio (USD)'); plt.show()

plt.figure(figsize=(9,6)); sns.heatmap(df[variables].corr(),annot=True,fmt='.2f',cmap='coolwarm')
plt.title('Matriz de correlación'); plt.xlabel('Variables'); plt.ylabel('Variables'); plt.show()


Los gráficos muestran que el precio es asimétrico y que existe una tendencia positiva entre superficie y precio, aunque con bastante dispersión. `Departamento` es el tipo de propiedad más frecuente.

# Fase 3 - Preparación de los Datos
## Missing values
Se revisan los porcentajes de nulos. Como cambian según `property_type` y `lat`/`lon` suelen faltar juntos, se consideran principalmente compatibles con **MAR**. También se prueba `KNNImputer` como en clases.


In [ ]:
resumen_nulos = pd.DataFrame({
    'cantidad_nulos': df.isna().sum(),
    'porcentaje_nulos': (df.isna().mean()*100).round(2)
})
resumen_nulos = resumen_nulos[resumen_nulos.cantidad_nulos>0].sort_values('porcentaje_nulos',ascending=False)
display(resumen_nulos)

plt.figure(figsize=(8,5)); plt.barh(resumen_nulos.index,resumen_nulos['porcentaje_nulos'])
plt.title('Porcentaje de valores faltantes'); plt.xlabel('Porcentaje (%)'); plt.ylabel('Variable')
plt.gca().invert_yaxis(); plt.show()

muestra_knn = df[['bathrooms','surface_total','surface_covered']].sample(1500,random_state=42)
knn = KNNImputer(n_neighbors=2,weights='uniform')
knn_resultado = pd.DataFrame(knn.fit_transform(muestra_knn),columns=muestra_knn.columns,index=muestra_knn.index)
display(pd.DataFrame({
    'media_original': muestra_knn.mean(),
    'media_knn': knn_resultado.mean(),
    'std_original': muestra_knn.std(),
    'std_knn': knn_resultado.std()
}).round(2))

df_limpio = df.copy()
for col in ['surface_total','surface_covered','bathrooms']:
    mediana = df_limpio.groupby('property_type',observed=False)[col].transform('median')
    df_limpio[col] = df_limpio[col].fillna(mediana)
for col in ['lat','lon']:
    mediana = df_limpio.groupby('l3',observed=False)[col].transform('median')
    df_limpio[col] = df_limpio[col].fillna(mediana)

print('Nulos restantes:', int(df_limpio.isna().sum().sum()))
print('Filas:', len(df_limpio))


KNN completa los nulos correctamente, pero para el tratamiento final se usa **mediana agrupada** porque es simple de interpretar y menos sensible a valores extremos. Se mantienen las 146.660 filas y quedan 0 nulos.

## Outliers
Se detectan con IQR y Z-score. No se eliminan automáticamente porque un precio o superficie alta puede corresponder a una propiedad real.


In [ ]:
outliers = []
for col in variables:
    q1,q3 = df_limpio[col].quantile([.25,.75]); iqr=q3-q1
    mascara = (df_limpio[col] < q1-1.5*iqr) | (df_limpio[col] > q3+1.5*iqr)
    z = np.abs(stats.zscore(df_limpio[col]))
    outliers.append([col,int(mascara.sum()),round(mascara.mean()*100,2),int((z>3).sum()),round((z>3).mean()*100,2)])
display(pd.DataFrame(outliers,columns=['variable','outliers_iqr','porcentaje_iqr','outliers_zscore','porcentaje_zscore']))

fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].boxplot(df_limpio['price'],vert=False); ax[0].set_title('Boxplot de price'); ax[0].set_xlabel('Precio (USD)'); ax[0].set_ylabel('Distribución')
ax[1].boxplot(df_limpio['surface_total'],vert=False); ax[1].set_title('Boxplot de surface_total'); ax[1].set_xlabel('Superficie total (m²)'); ax[1].set_ylabel('Distribución')
plt.tight_layout(); plt.show()

tratamiento=[]
for col in variables:
    limite=df_limpio[col].quantile(.99)
    cantidad=int((df_limpio[col]>limite).sum())
    max_antes=df_limpio[col].max()
    df_limpio[col]=df_limpio[col].clip(upper=limite)
    tratamiento.append([col,limite,cantidad,max_antes,df_limpio[col].max()])
display(pd.DataFrame(tratamiento,columns=['variable','limite_p99','ajustados','max_antes','max_despues']).round(2))


IQR detecta más casos que Z-score en las variables más asimétricas. Se usa el **percentil 99** para limitar solamente los extremos superiores sin eliminar filas.

## Normalización y estandarización
Después del tratamiento se revisa la asimetría. `rooms` y `bedrooms` se estandarizan con **StandardScaler**. Las variables todavía más asimétricas se normalizan con **MinMaxScaler**.


In [ ]:
print(df_limpio[variables].skew().round(3))

standard_cols=['rooms','bedrooms']
minmax_cols=['bathrooms','surface_total','surface_covered','price']

df_escalado=df_limpio.copy()
std=StandardScaler(); mm=MinMaxScaler()
df_escalado[[c+'_std' for c in standard_cols]]=std.fit_transform(df_limpio[standard_cols])
df_escalado[[c+'_minmax' for c in minmax_cols]]=mm.fit_transform(df_limpio[minmax_cols])

print('Antes:')
display(df_limpio[variables].agg(['mean','std','min','max']).T.round(3))
print('StandardScaler:')
display(df_escalado[[c+'_std' for c in standard_cols]].agg(['mean','std','min','max']).T.round(3))
print('MinMaxScaler:')
display(df_escalado[[c+'_minmax' for c in minmax_cols]].agg(['mean','std','min','max']).T.round(3))


Las columnas estandarizadas quedan con media cercana a 0 y desviación estándar cercana a 1. Las normalizadas quedan entre 0 y 1.

## Dataset final y exportación
Se corrigen los tipos de datos y se exporta el dataset preparado.


In [ ]:
df_final=df_escalado.copy()
for col in ['start_date','end_date','created_on']:
    df_final[col]=df_final[col].astype('datetime64[s]')
for col in ['l1','l2','l3','currency','property_type','operation_type']:
    df_final[col]=df_final[col].astype('category')

print('Dimensiones finales:',df_final.shape)
print('Valores nulos:',int(df_final.isna().sum().sum()))
print(df_final.dtypes)

df_final.to_csv('DS1-18-Datos-Properati-preparado.csv',index=False)
print('Dataset exportado correctamente.')


# Conclusiones
Se completaron las tres primeras fases de CRISP-DM. La superficie presenta una relación positiva con el precio, aunque existe dispersión y el precio no depende de una sola característica.

Los missing values se revisaron y se comparó `KNNImputer` con medianas agrupadas. Se eligieron medianas para conservar todas las filas y reducir el efecto de valores extremos.

Los outliers se detectaron con IQR y Z-score y se controlaron con percentil 99 sin eliminar registros. Finalmente se aplicaron `StandardScaler` y `MinMaxScaler` según la distribución observada y se corrigieron los tipos de datos.

El dataset final queda con **146.660 filas, 25 columnas y 0 valores nulos**, listo para una etapa posterior de modelado.
